# BIASED Retrieval Lab

Оригинальная лаборатория поиска: 14 вымышленных документов, 10 запросов, точные ID релевантности. Русская и английская версии. Материалы созданы для Фёдора Пахурова с помощью ИИ; это независимый учебный пример без сторонних датасетов и ноутбуков. Правила в документах вымышлены и не описывают условия BIASED School.

An original retrieval lab: 14 fictional documents, 10 queries and exact relevance IDs, in Russian and English. Prepared for Fedor Pakhurov with AI assistance. This independent teaching example contains no third-party dataset or notebook. The fictional policies are not BIASED School service terms.

**Запуск / Run:** Python 3, только стандартная библиотека / standard library only. Выполните ячейки сверху вниз / run the cells from top to bottom. Ничего скачивать не нужно / no downloads are needed.

**Ожидаемый результат / Expected result:** BM25 gives Hit@1 = Hit@3 = MRR = 0.8 in both languages. Queries q07 and q08 miss their relevant documents. Corpus and queries were constructed together to demonstrate errors; these numbers do not estimate production accuracy.

**Права / Rights:** Copyright 2026 Fedor Pakhurov, all rights reserved to the extent rights arise. No open-source or Creative Commons licence is granted. See the lab LICENSE on the notes website.


## 1. Данные внутри ноутбука / Embedded data

Прочитайте документы `d05` и `d14` и запросы `q08` и `q01`. Первый случай требует сопоставить перефразировку, второй содержит точный код ошибки. Метки используются только оценщиком, а не поиском.

Read documents `d05` and `d14`, and queries `q08` and `q01`. The first case needs a paraphrase match; the second contains an exact error code. Only the evaluator uses relevance labels; the retriever does not.


In [1]:
from collections import Counter
import json
import math
import re

DATASET_JSON = r'''{
  "id": "biased-retrieval-lab-v1",
  "title": "BIASED Retrieval Lab",
  "version": 1,
  "created": "2026-11-13",
  "provenance": {
    "creator": "Prepared for Fedor Pakhurov with AI assistance",
    "method": "Original fictional support documents, original queries and manually assigned relevance labels, written together for this teaching example. No third-party corpus, user messages or external notebook data were used.",
    "scope": "The policies describe a fictional learning workspace, not BIASED School's actual service terms.",
    "rights": "All rights reserved by Fedor Pakhurov to the extent rights arise. See LICENSE."
  },
  "documents": [
    {"id":"d01","text":{"en":"Sign-in code. A login code expires after fifteen minutes. Request a fresh code from the sign-in screen; the previous code then becomes invalid.","ru":"Код входа. Код действует пятнадцать минут. Запросите новый код на экране входа; предыдущий код после этого перестанет действовать."}},
    {"id":"d02","text":{"en":"Saved progress. Completed answers are stored in your account. Open the same account on another device to continue from the last saved answer.","ru":"Сохранённый прогресс. Завершённые ответы хранятся в аккаунте. Откройте тот же аккаунт на другом устройстве, чтобы продолжить с последнего сохранённого ответа."}},
    {"id":"d03","text":{"en":"Notebook restart. Restarting the notebook kernel clears variables from memory. Files saved in the workspace remain available after the restart.","ru":"Перезапуск ноутбука. Перезапуск ядра очищает переменные в памяти. Файлы, сохранённые в рабочей папке, остаются доступны после перезапуска."}},
    {"id":"d04","text":{"en":"PDF export. Open the lesson menu and choose Export PDF. The exported document contains the lesson text and diagrams, but no interactive controls.","ru":"Экспорт PDF. Откройте меню урока и выберите Экспорт PDF. Документ содержит текст и схемы урока, но не интерактивные элементы."}},
    {"id":"d05","text":{"en":"Offline reading. Download a lesson before travelling. The downloaded lesson text stays readable without a network connection; submitting answers requires reconnection.","ru":"Чтение офлайн. Скачайте урок перед поездкой. Скачанный текст доступен без подключения к сети; для отправки ответов нужно подключиться снова."}},
    {"id":"d06","text":{"en":"Theme settings. Choose a light or dark theme in Appearance. The selected theme is saved for this browser and does not change exported documents.","ru":"Настройка темы. Выберите светлую или тёмную тему в разделе Оформление. Выбор сохраняется в этом браузере и не меняет экспортированные документы."}},
    {"id":"d07","text":{"en":"Keyboard navigation. Press the Tab key to move to the next control. Press Shift and Tab together to move to the previous control.","ru":"Навигация клавиатурой. Клавиша Tab переносит фокус на следующий элемент. Нажмите Shift и Tab вместе, чтобы перейти к предыдущему элементу."}},
    {"id":"d08","text":{"en":"Video captions. Every workshop recording has a captions button. The transcript below the video also contains the spoken explanation as searchable text.","ru":"Субтитры видео. У каждой записи занятия есть кнопка субтитров. Расшифровка под видео содержит произнесённое объяснение в виде текста для поиска."}},
    {"id":"d09","text":{"en":"Project invitation. A project owner can invite a collaborator by email. The invitation grants access to the selected project, not to other projects in the account.","ru":"Приглашение в проект. Владелец проекта может пригласить участника по почте. Приглашение даёт доступ к выбранному проекту, но не к остальным проектам аккаунта."}},
    {"id":"d10","text":{"en":"Upload limit. A single uploaded file may be at most eight megabytes. Split a larger file into smaller files before uploading them separately.","ru":"Лимит загрузки. Размер одного загружаемого файла не должен превышать восемь мегабайт. Разделите большой файл на части и загрузите их отдельно."}},
    {"id":"d11","text":{"en":"Run quota. The workspace allows forty code runs per day. The counter resets at midnight UTC. Waiting for the reset does not delete saved files.","ru":"Квота запусков. В рабочей среде доступно сорок запусков кода в день. Счётчик сбрасывается в полночь UTC. Ожидание сброса не удаляет сохранённые файлы."}},
    {"id":"d12","text":{"en":"Shared folder. Files in a shared project folder are visible to every collaborator of that project. Keep private drafts in your personal folder.","ru":"Общая папка. Файлы общей папки проекта видны всем участникам этого проекта. Храните личные черновики в персональной папке."}},
    {"id":"d13","text":{"en":"Review reminders. Choose the days for review reminders in Notifications. Turning reminders off stops email messages and leaves your saved progress unchanged.","ru":"Напоминания о повторении. Выберите дни напоминаний в разделе Уведомления. Отключение напоминаний прекращает письма и не меняет сохранённый прогресс."}},
    {"id":"d14","text":{"en":"Error R17. This error means the output directory is read-only. Save the result in your personal folder and run the command again.","ru":"Ошибка R17. Эта ошибка означает, что папка для результата доступна только для чтения. Сохраните результат в персональную папку и запустите команду снова."}}
  ],
  "queries": [
    {"id":"q01","text":{"en":"R17 error output directory","ru":"Ошибка R17 папка результата"},"relevant_ids":["d14"],"case":"exact_identifier"},
    {"id":"q02","text":{"en":"upload file limit","ru":"Лимит загрузки файла"},"relevant_ids":["d10"],"case":"shared_terms"},
    {"id":"q03","text":{"en":"PDF lesson export","ru":"Экспорт PDF урока"},"relevant_ids":["d04"],"case":"shared_terms"},
    {"id":"q04","text":{"en":"notebook kernel restart variables","ru":"Перезапуск ядра ноутбука переменные"},"relevant_ids":["d03"],"case":"shared_terms"},
    {"id":"q05","text":{"en":"review reminders email","ru":"Напоминания о повторении письма"},"relevant_ids":["d13"],"case":"shared_terms"},
    {"id":"q06","text":{"en":"continue saved answers on another device","ru":"Продолжить сохранённые ответы на другом устройстве"},"relevant_ids":["d02"],"case":"shared_terms"},
    {"id":"q07","text":{"en":"hear nothing during playback","ru":"Звук пропал при просмотре"},"relevant_ids":["d08"],"case":"paraphrase_without_shared_terms"},
    {"id":"q08","text":{"en":"study aboard an airplane","ru":"Заниматься в самолёте"},"relevant_ids":["d05"],"case":"paraphrase_without_shared_terms"},
    {"id":"q09","text":{"en":"shared project folder collaborators","ru":"Общая папка проекта участники"},"relevant_ids":["d12"],"case":"nearby_distractor"},
    {"id":"q10","text":{"en":"code runs reset midnight","ru":"Запуски кода сброс полночь"},"relevant_ids":["d11"],"case":"shared_terms"}
  ]
}
'''
DATASET = json.loads(DATASET_JSON)
assert len(DATASET["documents"]) == 14
assert len(DATASET["queries"]) == 10
for doc_id in ["d05", "d14"]:
    doc = next(doc for doc in DATASET["documents"] if doc["id"] == doc_id)
    print(doc_id, doc["text"]["en"])
print("Documents:", len(DATASET["documents"]), "Queries:", len(DATASET["queries"]))


d05 Offline reading. Download a lesson before travelling. The downloaded lesson text stays readable without a network connection; submitting answers requires reconnection.
d14 Error R17. This error means the output directory is read-only. Save the result in your personal folder and run the command again.
Documents: 14 Queries: 10


## 2. Прозрачный BM25 / An inspectable BM25 baseline

Unicode-токены переводятся в нижний регистр; стемминга и удаления стоп-слов нет. Один уникальный токен запроса учитывается один раз. `k1=1.2`, `b=0.75`, положительный IDF: `log(1 + (N-df+0.5)/(df+0.5))`. Возвращаются только документы с положительным баллом; равные баллы разрешаются порядком ID.

Unicode word tokens are lowercased, without stemming or stopword removal. Each unique query term contributes once. The scoring parameters and positive IDF are listed above. Only positive-score documents are returned; score ties are broken by document ID. This is the same independent implementation as `retrieval-example.py`.


In [2]:
def tokens(text):
    return re.findall(r"[^\W_]+", text.casefold(), flags=re.UNICODE)

def rank(query, documents, locale, k1=1.2, b=0.75):
    counts = {doc["id"]: Counter(tokens(doc["text"][locale])) for doc in documents}
    lengths = {doc_id: sum(terms.values()) for doc_id, terms in counts.items()}
    mean_length = sum(lengths.values()) / len(documents)
    document_frequency = Counter(term for terms in counts.values() for term in terms)
    query_terms = set(tokens(query))
    scored = []
    for doc_id, terms in counts.items():
        score = 0.0
        for term in sorted(query_terms):
            tf = terms[term]
            df = document_frequency[term]
            idf = math.log(1 + (len(documents) - df + 0.5) / (df + 0.5))
            denominator = tf + k1 * (1 - b + b * lengths[doc_id] / mean_length)
            score += idf * tf * (k1 + 1) / denominator
        # Zero-score documents have no lexical evidence and are not returned.
        if score > 0:
            scored.append((doc_id, score))
    return sorted(scored, key=lambda item: (-item[1], item[0]))

def evaluate(dataset, locale):
    documents = dataset["documents"]
    document_ids = [doc["id"] for doc in documents]
    query_ids = [query["id"] for query in dataset["queries"]]
    if len(document_ids) != len(set(document_ids)) or len(query_ids) != len(set(query_ids)):
        raise ValueError("Document and query IDs must be unique")
    if not documents or not dataset["queries"]:
        raise ValueError("Documents and queries cannot be empty")
    rows = []
    for query in dataset["queries"]:
        relevant = set(query["relevant_ids"])
        if not relevant or not relevant.issubset(document_ids):
            raise ValueError("Each query must name existing relevant document IDs")
        ranking = rank(query["text"][locale], documents, locale)
        first = next((i for i, (doc_id, _) in enumerate(ranking, 1) if doc_id in relevant), None)
        rows.append({
            "query_id": query["id"],
            "relevant_ids": query["relevant_ids"],
            "first_relevant_rank": first,
            "top3": [{"id": doc_id, "score": round(score, 6)} for doc_id, score in ranking[:3]],
        })
    size = len(rows)
    return {
        "queries": rows,
        "metrics": {
            "hit_at_1": sum(row["first_relevant_rank"] == 1 for row in rows) / size,
            "hit_at_3": sum(row["first_relevant_rank"] is not None and row["first_relevant_rank"] <= 3 for row in rows) / size,
            "mrr": sum(1 / row["first_relevant_rank"] if row["first_relevant_rank"] else 0 for row in rows) / size,
        },
    }


## 3. Проверка всех запросов / Evaluate every query

Оценщик сравнивает точные ID. MRR использует позицию первого релевантного документа во всей возвращённой выдаче; пропуск даёт нулевой вклад. Hit@1 и Hit@3 применяют соответствующий порог.

The evaluator compares exact IDs. MRR uses the first relevant document in the full returned ranking; a miss contributes zero. Hit@1 and Hit@3 apply their respective cutoffs.


In [3]:
RESULTS = {locale: evaluate(DATASET, locale) for locale in ["en", "ru"]}
for locale, result in RESULTS.items():
    print(locale, json.dumps(result["metrics"], sort_keys=True))
    misses = [row["query_id"] for row in result["queries"] if row["first_relevant_rank"] is None]
    print("Missed queries:", ", ".join(misses))
    assert result["metrics"] == {"hit_at_1": 0.8, "hit_at_3": 0.8, "mrr": 0.8}
    assert misses == ["q07", "q08"]
print("Baseline output checks passed.")


en {"hit_at_1": 0.8, "hit_at_3": 0.8, "mrr": 0.8}
Missed queries: q07, q08
ru {"hit_at_1": 0.8, "hit_at_3": 0.8, "mrr": 0.8}
Missed queries: q07, q08
Baseline output checks passed.


## 4. Разбор пропуска / Inspect a miss

У `q08` нет общих токенов с нужным `d05`. В английской версии выдача пустая; в русской находятся посторонние документы с предлогом «в». Само увеличение k не возвращает документ с нулевым баллом при выбранном правиле выдачи.

Query `q08` has no shared tokens with the relevant `d05`. English returns no candidates; Russian returns irrelevant documents sharing the preposition "в". Increasing k cannot recover a zero-score document under this return policy.


In [4]:
query = next(q for q in DATASET["queries"] if q["id"] == "q08")
document = next(d for d in DATASET["documents"] if d["id"] == "d05")
for locale in ["en", "ru"]:
    overlap = set(tokens(query["text"][locale])) & set(tokens(document["text"][locale]))
    ranked = rank(query["text"][locale], DATASET["documents"], locale)
    print(locale, "shared tokens:", sorted(overlap), "top IDs:", [item[0] for item in ranked[:3]])
    assert overlap == set()
    assert "d05" not in [item[0] for item in ranked]


en shared tokens: [] top IDs: []
ru shared tokens: [] top IDs: ['d11', 'd03', 'd06']


## 5. Косинус на ручных векторах / Cosine on hand-chosen vectors

Это только арифметика: координаты выбраны вручную и не являются выводом обученной модели. Нельзя сравнивать полученные косинусы с качеством BM25 или считать их результатом dense retrieval.

These coordinates are hand-chosen arithmetic examples, not model outputs. They do not measure dense retrieval or establish an improvement over the lexical baseline.


In [5]:
def cosine(left, right):
    dot = sum(a * b for a, b in zip(left, right))
    return dot / (math.sqrt(sum(a * a for a in left)) * math.sqrt(sum(b * b for b in right)))

query_vector = [1.0, 0.0]
example_vectors = {"d05": [0.8, 0.6], "d10": [0.0, 1.0]}
for doc_id, vector in example_vectors.items():
    print(doc_id, "illustrative cosine:", cosine(query_vector, vector))
assert math.isclose(cosine(query_vector, example_vectors["d05"]), 0.8)
assert math.isclose(cosine(query_vector, example_vectors["d10"]), 0.0)


d05 illustrative cosine: 0.8
d10 illustrative cosine: 0.0


## 6. Попробуйте свой запрос / Try another query

Измените `QUERY` и изучите ранжирование. Если хотите измерять качество на новых запросах, сначала задайте релевантные ID независимо от выдачи. Изменение корпуса требует повторного подсчёта частот и длины документов — эта функция пересчитывает их при каждом вызове.

Edit `QUERY` and inspect the ranking. For new evaluation queries, define relevant IDs independently of the returned results. Corpus changes require recomputing frequencies and lengths; this small implementation does so on every call.


In [6]:
LOCALE = "en"
QUERY = "PDF export"
for doc_id, score in rank(QUERY, DATASET["documents"], LOCALE)[:3]:
    print(doc_id, round(score, 6))


d04 6.337627


## Что проверить дальше / What to test next

- Проверьте новую формулировку запроса с заранее заданным релевантным ID. / Test a new query wording with a relevance label fixed in advance.
- Проверьте, что переранжирование меняет только порядок переданных кандидатов. / Check that reranking can reorder candidates, not create a missing one.
- Если добавляете настоящий энкодер, зафиксируйте модель, обработку и индекс и выполните отдельную оценку. / If adding a learned encoder, record the model, preprocessing and index, then run a separate evaluation.

### Источники алгоритмов / Algorithm references

Original lab text, data and code were independently created; cited publications are conceptual references only.

- [Robertson and Zaragoza, 2009: BM25 and Beyond](https://doi.org/10.1561/1500000019)
- [Karpukhin et al., 2020: Dense Passage Retrieval](https://aclanthology.org/2020.emnlp-main.550/)
- [Nogueira and Cho, 2019: Passage Re-ranking with BERT](https://arxiv.org/abs/1901.04085)
